# Agent Actions

**Important:**
- If you are just exploring Mesa and want the fastest way to execute the code we recommend executing this tutorial online in a Colab notebook. [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mesa/mesa/blob/main/docs/tutorials/4_agent_actions.ipynb) or if you do not have a Google account you can use [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/mesa/mesa/main?labpath=docs%2Ftutorials%2F4_agent_actions.ipynb) (This can take 30 seconds to 5 minutes to load)
- If you are running locally, please ensure you have the latest Mesa version installed.

## Tutorial Description

In the [event scheduling tutorial](3_event_scheduling.ipynb) you scheduled events at
specific moments in time. An **Action** builds on that: it represents something an agent
*does over time*, with a duration, requirements that can fail it, and a priority that
decides whether it can be interrupted.

This tutorial covers `mesa.experimental.actions`, which lets you write agent behaviour
without a polling `step()` method.

By the end of this tutorial you will know how to:
- Give an agent the ability to perform actions with the `HasActions` mixin
- Define an action with a duration and lifecycle hooks
- Gate an action with start and completion requirements, and handle failure
- Interrupt a running action and let `priority` decide who yields
- React when an action ends using the `on_idle` wake hook
- Combine the two things that wake an agent: its body becoming free, and the world
  crossing a line it declared in advance
- Resume an action that was interrupted partway through

> **Experimental API.** Everything in `mesa.experimental` can change without a
> deprecation cycle. This tutorial was written against Mesa's `main` branch.

### IN COLAB? - Run the next cell

In [ ]:
%pip install --quiet mesa[rec]

### Import Dependencies

In [ ]:
from mesa import Agent, Model
from mesa.experimental.actions import Action, HasActions
from mesa.experimental.mesa_signals import HasEmitters
from mesa.experimental.states import ContinuousState, Threshold

## Why Actions?

A classic agent-based model puts behaviour in `step()`, which runs for every agent on
every tick. That has two costs. Every agent is asked what it is doing even when nothing
has changed for it, and anything that takes time has to be hand tracked with counters
like `self.grazing_for += 1`.

An `Action` makes the duration explicit. The agent says "I am grazing, it takes five time
units", and Mesa schedules exactly one event for the moment it finishes. Nothing polls in
between.

## Your First Action

An action is a class with a duration and some hooks. The most common one is
`on_complete()`, which applies the effect when the action finishes.

Agents opt into actions with the `HasActions` mixin.

In [ ]:
class Forage(Action):
    def __init__(self, sheep):
        super().__init__(sheep, duration=5.0)

    def on_complete(self):
        self.agent.energy += 30


class Sheep(HasActions, Agent):
    def __init__(self, model, energy=50):
        super().__init__(model)
        self.energy = energy


model = Model(rng=42)
sheep = Sheep(model)
sheep.start_action(Forage(sheep))

print(f"t={model.time}  busy={sheep.is_busy}  energy={sheep.energy}")
model.run_for(5)
print(f"t={model.time}  busy={sheep.is_busy}  energy={sheep.energy}")

Note the class declaration:

```python
class Sheep(HasActions, Agent):
```

**The mixin comes first.** This matters: `Agent.remove()` does not call
`super().remove()`, so if you write `class Sheep(Agent, HasActions)` the action cleanup
never runs and a removed agent leaves its completion event behind. Mesa uses this same
pattern elsewhere, for instance `CellAgent` is `class CellAgent(Agent, HasCell, BasicMovement)`.

To combine actions with space, put `HasActions` in front of the spatial base class:

```python
class Wolf(HasActions, CellAgent):
    ...
```

## The Action Lifecycle

An action moves through `ActionState` values: `PENDING` before it starts, `ACTIVE` while
it runs, and then one of `COMPLETED`, `INTERRUPTED` or `FAILED`.

While an action is active, `progress`, `remaining_time` and `elapsed_time` are computed
live from the clock, so you can inspect them at any moment without storing anything.

In [ ]:
model = Model(rng=42)
sheep = Sheep(model)
forage = Forage(sheep)

print(f"before start : {forage.state.name:11} progress={forage.progress:.0%}")

sheep.start_action(forage)
print(f"just started : {forage.state.name:11} progress={forage.progress:.0%}")

model.run_for(2)
print(
    f"t={model.time}        : {forage.state.name:11} progress={forage.progress:.0%} "
    f"remaining={forage.remaining_time}"
)

model.run_for(3)
print(f"t={model.time}        : {forage.state.name:11} progress={forage.progress:.0%}")

## Requirements and Failure

An action can be gated by requirements, each a callable that receives the agent and
returns a bool:

- **`start_requirements`** decide whether the action begins at all.
- **`completion_requirements`** decide whether the effect lands when it finishes.

If a requirement does not hold, the action moves to `FAILED` and `on_fail()` runs instead
of `on_start()` or `on_complete()`.

The two lists answer different questions. A contested resource should be *claimed at the
start*, inside `on_start()`, so nobody else can take it. The completion list is for state
nobody can reserve, such as a market still being open when a trade settles.

Nothing is re-checked in between. An action that must abort mid-flight is a job for a
`Threshold`, which we get to below.

In [ ]:
patch = {"servings": 1}


class Graze(Action):
    def __init__(self, sheep):
        super().__init__(
            sheep,
            duration=3.0,
            name="Graze",
            start_requirements=lambda a: patch["servings"] > 0,
        )

    def on_start(self):
        patch["servings"] -= 1  # claimed, so nobody else can take it

    def on_complete(self):
        self.agent.energy += 30

    def on_fail(self):
        print(f"  sheep {self.agent.unique_id}: no serving left, looking elsewhere")


model = Model(rng=42)
first, second = Sheep(model), Sheep(model)

a1 = first.start_action(Graze(first))
a2 = second.start_action(Graze(second))

print(f"first  : {a1.state.name}")
print(f"second : {a2.state.name}  has_failed={a2.has_failed}")

model.run_for(3)
print(f"energies: first={first.energy}  second={second.energy}")

## Interruption, Priority and Resumption

`interrupt_for()` asks the agent to drop what it is doing and start something else. The
agent decides through `should_interrupt(current, incoming)`, which by default compares
priorities:

```python
return current.interruptible and incoming.priority >= current.priority
```

Override it to encode a different policy. An action marked `interruptible=False` refuses
regardless, and `cancel_action()` is the way to force a stop.

An interrupted action keeps its progress and becomes resumable.

In [ ]:
class Flee(Action):
    def __init__(self, sheep):
        super().__init__(
            sheep, duration=2.0, name="Flee", priority=10.0, interruptible=False
        )


patch["servings"] = 5
model = Model(rng=42)
sheep = Sheep(model)

graze = sheep.start_action(Graze(sheep))
model.run_for(1)

started = sheep.interrupt_for(Flee(sheep))
print(f"interrupted : {started}")
print(
    f"graze       : {graze.state.name} at {graze.progress:.0%}, resumable={graze.is_resumable}"
)
print(f"now doing   : {sheep.current_action.name}")

## The Wake Contract: `on_idle`

When an action ends and nothing has replaced it, the agent's `on_idle(previous)` hook
fires. This is how an agent decides what to do next, and it is the reason a model built
on actions needs no `step()` at all.

`on_idle` fires for *every* way an action can end: completion, interruption,
cancellation, and failure. The `previous` argument is the action that just ended, in its
final state.

The wake is dispatched as a zero-delay, low-priority event rather than a direct call, so
every agent finishing at time *t* decides after all completions at *t* have run.

In [ ]:
class Worker(HasActions, Agent):
    def __init__(self, model):
        super().__init__(model)
        self.log = []

    def on_idle(self, previous):
        if previous is not None:
            self.log.append(
                f"t={self.model.time}: finished {previous.name} ({previous.state.name})"
            )
        self.start_action(Action(self, duration=1.0, name="Task"))


model = Model(rng=42)
worker = Worker(model)
worker.start_action(Action(worker, duration=1.0, name="Task"))

model.run_for(3.5)
for line in worker.log:
    print(line)

## The Two Wake Sources

An agent's decision making wakes on exactly two triggers:

1. **Its body became free.** An action ended, so `on_idle` fires.
2. **The world crossed a line it declared in advance.** A `Threshold` on a
   `ContinuousState` fires its callback.

There is no queue to drain and no polling loop. Everything else, including the model's
`step()`, is optional.

The example below uses both. Thirst rises continuously as a `ContinuousState`, a
`Threshold` interrupts grazing when the sheep gets too thirsty, and `on_idle` decides
what happens once drinking is done.

In [ ]:
class Drink(Action):
    def __init__(self, sheep):
        super().__init__(
            sheep, duration=2.0, name="Drink", priority=10.0, interruptible=False
        )

    def on_complete(self):
        self.agent.thirst = 0.0


class ThirstySheep(HasActions, Agent, HasEmitters):
    thirst = ContinuousState(fallback_value=0.0, rate=1.0)
    _thirsty = Threshold(
        state=thirst, limit=8.0, callback="got_thirsty", direction="rising"
    )

    def __init__(self, model):
        super().__init__(model)
        self.energy = 50
        self.log = []
        self.unfinished = None
        self.thirst = 0.0  # assigning the state binds the threshold

    # wake source 2: the world crossed a line
    def got_thirsty(self):
        self.log.append(f"t={self.model.time}: thirsty, interrupting")
        self.unfinished = self.current_action  # capture before interrupting
        self.interrupt_for(Drink(self))

    # wake source 1: the body became free
    def on_idle(self, previous):
        if self.unfinished is not None and self.unfinished.is_resumable:
            resuming, self.unfinished = self.unfinished, None
            self.log.append(
                f"t={self.model.time}: resuming {resuming.name} "
                f"at {resuming.progress:.0%}"
            )
            self.start_action(resuming)
        else:
            self.log.append(f"t={self.model.time}: starting a fresh Graze")
            self.start_action(Graze(self))


patch["servings"] = 99
model = Model(rng=42)
sheep = ThirstySheep(model)
sheep.start_action(Graze(sheep))

model.run_for(16)
for line in sheep.log:
    print(line)
print(f"energy={sheep.energy}  thirst={sheep.thirst:.1f}")

## Recipe: Resume the Remainder

The example above contains a pattern worth calling out, because it is the one thing
people reach for first and it is deliberately **not** automatic.

`on_idle(previous)` hands you the action that *just* ended. After an interruption that is
the interrupting action (`Drink`), not the one that was interrupted (`Graze`). Mesa does
not remember the interrupted action for you: whether to resume it is a modelling
decision, not a framework one.

So the model remembers it:

```python
def got_thirsty(self):
    self.unfinished = self.current_action     # capture before interrupting
    self.interrupt_for(Drink(self))

def on_idle(self, previous):
    if self.unfinished is not None and self.unfinished.is_resumable:
        resuming, self.unfinished = self.unfinished, None
        self.start_action(resuming)           # picks up the remaining 20%
    else:
        self.start_action(Graze(self))
```

Two details matter:

- **Capture before interrupting.** `interrupt_for()` clears `current_action`, so reading
  it afterwards gives you `None`.
- **Check `is_resumable`.** It is True only for an action that was interrupted with
  progress left to run. A completed or failed action cannot be restarted, and passing one
  to `start_action()` raises.

Resuming runs only the remaining duration. In the output above the graze was interrupted
two thirds of the way through, so it needed just one more time unit to finish rather than
a full three.

## Summary

- **`HasActions`** gives an agent the ability to perform actions. Put the mixin first:
  `class Sheep(HasActions, Agent)`.
- An **`Action`** has a duration and the hooks `on_start`, `on_resume`, `on_complete`,
  `on_interrupt(progress)` and `on_fail`.
- **`start_requirements`** gate the attempt, **`completion_requirements`** gate the
  effect, and a broken requirement moves the action to `FAILED`.
- **`interrupt_for()`** asks `should_interrupt()`, whose default lets a higher or equal
  `priority` win. `interruptible=False` refuses, `cancel_action()` forces.
- **`on_idle(previous)`** fires whenever an action ends and nothing replaced it. This is
  where the agent decides what to do next.
- An agent wakes from exactly two sources: `on_idle` and a `Threshold` crossing.
- Resuming an interrupted action is a modelling decision. Remember the action yourself
  and check `is_resumable`.

## Next Steps

Check out the [adding space tutorial](5_adding_space.ipynb) on how to add a space to your Mesa model.

For a larger worked example of the event driven style, have a look at the [Tram Route Model](https://mesa.readthedocs.io/latest/examples/experimental/tram_model.html), built entirely on `ContinuousState` and `Threshold`, where every transition is an analytically solved crossing and the model has no agent `step()` logic at all.
